In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import astropy.coordinates as coord
import astropy.units as u
import astropy.io.fits as fits
from astropy.table import Table,join,vstack,unique,QTable
import sys
from urllib.parse import urljoin
from scipy.ndimage import gaussian_filter
import seaborn as sns
from scipy.special import erf
from scipy.optimize import curve_fit
from scipy.spatial import KDTree

if './SelfCalGroupFinder/py/' not in sys.path:
    sys.path.append('./SelfCalGroupFinder/py/')
from pyutils import *
from dataloc import *
from bgs_helpers import *
import groupcatalog as gc
from nnanalysis import *
from plotting import *
from footprintmanager import FootprintManager

%load_ext autoreload
%autoreload 2

pd.set_option('display.max_columns', None)

In [ ]:
def get_max_observable_z(abs_mags, fluxlimit):
    d_l = (10 ** ((fluxlimit - abs_mags + 5) / 5)) / 1e6 # luminosity distance in Mpc
    return z_at_value(get_cosmology().luminosity_distance, d_l*u.Mpc) # TODO what cosmology to use?


In [ ]:
def make_adaptive_density_bins(data, n_bins, n_tail, alpha, limit):
    """
    Widths go as 1/f(x)^alpha (alpha=0.5: sqrt-density spacing).
    Implemented by spacing edges evenly in cumulative(f^alpha) space.
    Tail extensions use the edge bin spacing for a smooth cutoff to zero.
    """
    N = len(data)
    
    # Pilot density over the covered range (exclude extreme outliers)
    p_lo, p_hi = np.quantile(data, [limit, 1 - limit])
    n_pilot = min(2000, N // 50)
    pilot_edges = np.linspace(p_lo, p_hi, n_pilot + 1)
    pilot_counts, _ = np.histogram(data, bins=pilot_edges)
    pilot_density = pilot_counts / np.diff(pilot_edges) / N
    # Floor: avoid pathological zero regions swallowing all the tail budget
    #floor = pilot_density[pilot_density > 0].min() * 0.01
    #pilot_density = np.maximum(pilot_density, floor)
    
    # Cumulative of f^alpha → spacing evenly in this = widths ∝ 1/f^alpha
    dx = np.diff(pilot_edges)
    cumulative = np.concatenate([[0], np.cumsum(pilot_density**alpha * dx)])
    total = cumulative[-1]
    
    target = np.linspace(0, total, n_bins + 1)
    adaptive_edges = np.unique(np.interp(target, cumulative, pilot_edges))
    
    # Extend tails with uniform bins at the edge. Capture some of the rarest halos. Mostly 0's out here.
    dl = np.median(np.diff(adaptive_edges[:10]))
    dr = np.median(np.diff(adaptive_edges[-10:]))
    left  = adaptive_edges[0]  - np.arange(n_tail, 0, -1) * dl
    right = adaptive_edges[-1] + np.arange(1, n_tail + 1)  * dr
    
    return np.concatenate([left, adaptive_edges, right])

In [ ]:
fits = fitsio.FITS(BGS_Y3_ANY_FULL_FILE)
hdu = fits[1] 
hdu.get_colnames()

In [ ]:
fulldat = fitsio.read(BGS_Y3_ANY_FULL_FILE)
print(len(fulldat), 'full data rows read from', BGS_Y3_ANY_FULL_FILE)

# GET DATA FROM GROUP CATALOG AND MERGED FILE instead of raw LSScat
# We've already made a bunch of quality cuts on this.
path = '/global/cfs/cdirs/desi/users/ianw89/groupcatalogs/BGS_Y3/v0.8/GROUP_CATALOG_BGS_Y3_1PASS_v0.8.fits'
grpcat = Table(fitsio.read(path, columns=['TARGETID', 'Z_ASSIGNED_FLAG', 'ABS_MAG_R', 'G_R', 'IS_SAT']))
print(len(grpcat), 'group catalog rows read from', path)

sel = grpcat['Z_ASSIGNED_FLAG'] == 0 # DESI spectroscopic data only
sel &= ~grpcat['IS_SAT'] # Centrals only
grpcat = grpcat[sel].reset_index(drop=True)
grpcat.drop(columns=['Z_ASSIGNED_FLAG', 'IS_SAT'], inplace=True)
print(len(grpcat), 'group catalog rows after cuts (DESI spec + centrals only)')

# Merge with fulldat, inner join
fulldat = join(fulldat, grpcat, keys=['TARGETID'], join_type="inner")
print(len(fulldat), 'full data rows after merging with group catalog')


# Now read in merged file and get extra columns we need (no need yet)
#extra = Table(fitsio.read('/global/cfs/cdirs/desi/users/ianw89/private/DATA/BGS_LOA/ian_BGS_Y3_merged.fits', columns=['TARGETID', 'c9050']))
#fulldat = join(fulldat, extra, keys=['TARGETID'], join_type="inner")

assert ~np.isnan(grpcat['ABS_MAG_R']).all() & ~np.isnan(grpcat['G_R']).all()

In [ ]:
# Testing adaptive binning

# group cat
path = '/global/cfs/cdirs/desi/users/ianw89/groupcatalogs/BGS_Y3/v0.8/GROUP_CATALOG_BGS_Y3_1PASS_v0.8.fits'
df = Table.read(path).to_pandas()

df = df.loc[~df['IS_SAT'], :].reset_index(drop=True)
df.drop(columns=['RA', 'DEC', 'L_GAL', 'L_TOT', 'P_SAT', 'N_SAT', 'WEIGHT', 'IGRP', 'QUIESCENT', 'APP_MAG_R', 'IS_SAT'], inplace=True)
print(len(df))

# Now read in merged file and get extra columns we need
table = Table(fitsio.read(IAN_BGS_Y3_MERGED_FILE_LOA, columns=['TARGETID', 'c9050'])).to_pandas()
df = df.merge(table, on='TARGETID', how='inner')
print(len(df))

# This binnings strategy looks reasonable

mask = ~np.isnan(df['ABS_MAG_R']) & ~np.isnan(df['G_R']) & ~np.isnan(df['c9050'])

mags = df['ABS_MAG_R'][mask]
gr = df['G_R'][mask]
c9050 = df['c9050'][mask]
print(len(mags))

mag_bins = make_adaptive_density_bins(mags, n_bins=10, n_tail=0, alpha=0.5, limit=0.01)

# Now make g-r bins for each of those mag bins
for i in range(len(mag_bins)-1):
    mag_mask = (mags >= mag_bins[i]) & (mags < mag_bins[i+1])
    gr_in_bin = gr[mag_mask]
    gr_bins = make_adaptive_density_bins(gr_in_bin, n_bins=5, n_tail=0, alpha=0.5, limit=0.01)

    for j in range(len(gr_bins)-1):
        gr_mask = (gr_in_bin >= gr_bins[j]) & (gr_in_bin < gr_bins[j+1])
        c9050_in_bin = c9050[mag_mask][gr_mask]
        cbins = make_adaptive_density_bins(c9050_in_bin, n_bins=5, n_tail=0, alpha=0.5, limit=0.01)
        print(f"Mag bin {i}: np{mag_bins[i]:.2f} to {mag_bins[i+1]:.2f}, g-r bin {j}: {gr_bins[j]:.2f} to {gr_bins[j+1]:.2f}, c9050 bins: {cbins}, counts: {np.histogram(c9050_in_bin, bins=cbins)[0]}")

### Test subsample catalogs written

In [ ]:
basepath = '/global/cfs/cdirs/desi/users/ianw89/clustering712/DA2/LSS/loa-v1/LSScats/v1.1/'
datpath = basepath + 'BGS_BRIGHT_CEN_mag-21.4880to-21.0910_gr0.3945to0.6139_clustering.dat.fits'
ranpath = basepath + 'BGS_BRIGHT_CEN_mag-21.4880to-21.0910_gr0.3945to0.6139_0_clustering.ran.fits'

In [ ]:
dat = Table(fitsio.read(datpath))
ran = Table(fitsio.read(ranpath))

print(len(dat), 'data rows read from', datpath)
print(len(ran), 'random rows read from', ranpath)

In [ ]:
# print columns
print(dat.colnames)
print(ran.colnames)

In [ ]:
# Show z histogram for this bin
plt.figure(figsize=(8, 5))
plt.hist(dat['Z'], bins=30, alpha=0.5, label='Data', density=True, histtype='step')
plt.hist(ran['Z'], bins=30, alpha=0.5, label='Randoms', density=True, histtype='step')
plt.xlabel('Redshift (z)')
plt.ylabel('Normalized Counts')

# TODO drop because of 19.54 vs 19.5 maybe?